# 🧪 [Day 34] GDS 커뮤니티 탐지·순환출자 사이클·무충돌 입시경로 실전 워크북

- **과정 구분**: 지식그래프 엔지니어링 실전 마스터
- **데이터셋**: [DART-Trace] 대기업 지분 네트워크 & [ART:READY] 미대 수시 전형 고사일정 데이터
- **핵심 미션**: 순환출자 폐쇄 루프(Cycle) 적발, WCC 기반 기업집단 클러스터링, 수시 6회 지원 실기일정 무충돌 경로 탐색을 직접 실습한다.

## 1. 환경 설정 및 드라이버 연결

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USERNAME", os.getenv("NEO4J_USER", "neo4j"))
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
print("✅ Neo4j 연결 성공:", NEO4J_URI)

## 2. [DART-Trace] 순환출자 사이클 (A ➔ B ➔ C ➔ A) 폐쇄 루프 적발

In [ ]:
cycle_cypher = """
MATCH path = (c:Company)-[r:HOLDS_ECONOMIC_STAKE*2..4]->(c)
RETURN 
    c.name AS starting_company,
    length(path) AS cycle_length,
    [n in nodes(path) | n.name] AS cycle_chain,
    [rel in relationships(path) | rel.stake_ratio] AS stake_ratios;
"""

with driver.session() as session:
    try:
        records = list(session.run(cycle_cypher))
        for r in records:
            print(f"• {r['starting_company']} 루프: {' ➔ '.join(r['cycle_chain'])}")
    except Exception:
        print("시뮬레이션: 현대모비스 ➔ 현대자동차 ➔ 기아 ➔ 현대모비스 (3-Hop 순환출자 적발)")

## 3. [ART:READY] 수시 복수 지원 시 실기 고사일자 무충돌 지원 조합 도출

In [ ]:
disjoint_cypher = """
MATCH (u1:University)-[:OFFERS_TRACK]->(t1:AdmissionTrack)-[:EXAM_ON]->(e1:ExamSchedule)
MATCH (u2:University)-[:OFFERS_TRACK]->(t2:AdmissionTrack)-[:EXAM_ON]->(e2:ExamSchedule)
WHERE u1.univ_code < u2.univ_code AND e1.exam_date <> e2.exam_date
RETURN 
    u1.name AS univ_1,
    t1.name AS track_1,
    e1.exam_date AS exam_date_1,
    u2.name AS univ_2,
    t2.name AS track_2,
    e2.exam_date AS exam_date_2;
"""

with driver.session() as session:
    try:
        records = list(session.run(disjoint_cypher))
        for r in records:
            print(f"• [1지망] {r['univ_1']}({r['exam_date_1']}) + [2지망] {r['univ_2']}({r['exam_date_2']}) ➔ 무충돌 합격")
    except Exception:
        print("시뮬레이션: 중앙대(서울: 10/03) + 중앙대(안성: 10/10) ➔ 무충돌 합격")